# Part 2 - Build the time-resolved multilayer graph

This notebook builds the two-aspect `mechanism x time` graph and writes the
snapshot the rest of the series reloads. Run `01_setup` first.

The build reuses UC2's heterogeneous sources but places each entity in *time*
instead of *structure*. The **responsive layers** (1h..96h) hold the dynamics:
a signaling or regulatory edge exists in layer `t` only when both its endpoints
are responsive at `t` (pure 0-hop restriction, from `01`). The **baseline layer**
(`0h`) holds the time-invariant scaffold: complexes and the metabolic model.
Two coupling families then bind the layers - diagonal *time* coupling
(the same entity across consecutive timepoints) and *translation* coupling
(a gene to its protein within a timepoint).

In [1]:
import contextlib
import io

import polars as pl
import requests

import annnet as an
import uc2b_common as uc

responsive = pl.read_parquet(str(uc.RESPONSIVE))
measured_syms = set(pl.read_parquet(str(uc.MEASURED))["symbol"])
resp_at = uc.responsive_by_time(responsive.to_pandas())  # {time: set(symbol)}

G = an.AnnNet(directed=True)
G.history.enable(True)
G.history.snapshot("init")
G.layers.set_aspects(uc.ASPECTS, uc.ELEM_LAYERS)
print("aspects:", G.layers.list_aspects())
print("layers :", G.layers.list_layers())

aspects: ('mechanism', 'time')
layers : {'mechanism': ['metabolic', 'regulatory', 'signaling'], 'time': ['0h', '12h', '1h', '24h', '48h', '72h', '96h']}


## Responsive time layers - signaling and regulatory

`from_omnipath` fetches the OmniPath signaling network; DoRothEA (confidence A, B)
supplies the regulatory network. For each mechanism, `add_time_layers` walks the
six timepoints and, at each, keeps only the edges whose endpoints are both
responsive then, placing those supra-nodes at `(mechanism, t)`. The same protein
therefore appears once per timepoint it responds in - the multilayer expansion, so
the time signal lives in *which layers a vertex belongs to* and in the edges'
coordinates. (AnnNet keeps one attribute record per `vertex_id`, so the stored
`best_logFC` is a summary; the time-resolved values stay in `responsive.parquet`,
which the queries and the GNN read directly.)

In [2]:
with contextlib.redirect_stdout(io.StringIO()):
    pkn = an.from_omnipath(dataset="omnipath", query={"genesymbols": True},
                           source_col="source_genesymbol", target_col="target_genesymbol",
                           edge_attr_cols=["is_stimulation", "is_inhibition"],
                           load_vertex_annotations=False)
sig = pkn.views.edges().to_pandas()
sig_pairs = [(r.source, r.target, uc.sign_of(r.is_stimulation, r.is_inhibition))
             for r in sig.itertuples(index=False)]

dor = pl.read_csv(str(uc.DATA / "dorothea.tsv"), separator="\t", infer_schema_length=5000)
reg_pairs = [(r["source_genesymbol"], r["target_genesymbol"],
              uc.sign_of(r["is_stimulation"], r["is_inhibition"])) for r in dor.iter_rows(named=True)]


def add_time_layers(mechanism, pairs, prefix, kind):
    """Add the responsive-gated supra-nodes and edges for one mechanism, per timepoint.

    Returns {(symbol, time)} of the supra-nodes placed, for the coupling step."""
    present = set()
    for t in uc.TIMES:
        r = resp_at.get(t, set())
        kept = [(a, b, w) for a, b, w in pairs if a in r and b in r]
        lfc = dict(zip(responsive.filter(pl.col("time") == t)["symbol"],
                       responsive.filter(pl.col("time") == t)["best_logFC"]))
        verts = {v for a, b, _ in kept for v in (a, b)}
        G.add_vertices([{"vertex_id": f"{prefix}{v}", "gene_symbol": v, "kind": kind,
                         "best_logFC": float(lfc.get(v, 0.0))} for v in verts], layer=(mechanism, t))
        G.add_edges([{"source": (f"{prefix}{a}", (mechanism, t)),
                      "target": (f"{prefix}{b}", (mechanism, t)),
                      "weight": w, "edge_kind": mechanism} for a, b, w in kept],
                    default_edge_directed=True)
        present |= {(v, t) for v in verts}
    return present

sig_present = add_time_layers(uc.MECH_SIGNALING, sig_pairs, "prot:", uc.KIND_PROTEIN)
reg_present = add_time_layers(uc.MECH_REGULATORY, reg_pairs, "gene:", uc.KIND_GENE)
G.history.snapshot("after_response")

print(f"{'t':>5} {'sig_V':>6} {'sig_E':>6} {'reg_V':>6} {'reg_E':>6}")
for t in uc.TIMES:
    sv = len(G.layers.layer_vertex_set((uc.MECH_SIGNALING, t)))
    rv = len(G.layers.layer_vertex_set((uc.MECH_REGULATORY, t)))
    se = sum(1 for _, (s, tt, _) in G.edge_definitions.items()
             if isinstance(s, tuple) and s[1] == (uc.MECH_SIGNALING, t))
    re = sum(1 for _, (s, tt, _) in G.edge_definitions.items()
             if isinstance(s, tuple) and s[1] == (uc.MECH_REGULATORY, t))
    print(f"{t:>5} {sv:>6} {se:>6} {rv:>6} {re:>6}")

/home/l1boll/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    t  sig_V  sig_E  reg_V  reg_E
   1h    107    137    116    158
  12h    815   1732    513    838
  24h   1131   2668    750   1346


  48h   1651   4885   1063   2044
  72h   1955   5988   1221   2184
  96h   2984  10920   1840   3592


## Interlayer coupling - time and translation

Two coupling families bind the layers. **Time coupling** links the same entity at
consecutive timepoints, `(v, mech, t_i) -> (v, mech, t_{i+1})`, wherever it is
responsive in both - a real diagonal edge in the `time` aspect, the structure UC2's
static design could only assert. **Translation coupling** links a responsive gene
to its protein within a timepoint, `(gene, regulatory, t) -> (prot, signaling, t)`,
bridging the two mechanisms.

In [3]:
def couple_time(present, mechanism, prefix):
    n = 0
    for ti, tj in zip(uc.TIMES, uc.TIMES[1:]):
        at_i = {v for v, t in present if t == ti}
        shared = at_i & {v for v, t in present if t == tj}
        G.add_edges([{"edge_id": f"time:{mechanism}:{v}:{ti}->{tj}", "edge_kind": "coupling_time",
                      "weight": 1.0, "source": (f"{prefix}{v}", (mechanism, ti)),
                      "target": (f"{prefix}{v}", (mechanism, tj))} for v in shared],
                    default_edge_directed=True)
        n += len(shared)
    return n

n_time = couple_time(sig_present, uc.MECH_SIGNALING, "prot:") + \
         couple_time(reg_present, uc.MECH_REGULATORY, "gene:")

n_trans = 0
for t in uc.TIMES:
    both = {v for v, tt in reg_present if tt == t} & {v for v, tt in sig_present if tt == t}
    G.add_edges([{"edge_id": f"trans:{v}:{t}", "edge_kind": "coupling_translation", "weight": 1.0,
                  "source": (f"gene:{v}", (uc.MECH_REGULATORY, t)),
                  "target": (f"prot:{v}", (uc.MECH_SIGNALING, t))} for v in both],
                default_edge_directed=True)
    n_trans += len(both)
G.history.snapshot("after_coupling")
print(f"time-coupling edges       : {n_time:,}")
print(f"translation-coupling edges: {n_trans:,}")

time-coupling edges       : 7,660
translation-coupling edges: 3,322


## Baseline scaffold - complexes and the metabolic model

Complexes and metabolism are time-invariant, so they live in the `0h` baseline
layer. Each OmniPath complex whose subunits are all in the measured universe
becomes one undirected hyperedge - a single incidence-matrix column, not a clique.
One `from_sbml` call reads Human-GEM: its reactions become signed hyperedges and
its compartments become slices. `from_sbml` adds metabolite and boundary vertices
without a `kind`, so this step tags them for the later `to_pyg` export.

In [4]:
cpx = pl.read_csv(str(uc.DATA / "omnipath_complexes.tsv"), separator="\t", infer_schema_length=5000)
complex_specs, members = [], set()
for row in cpx.iter_rows(named=True):
    subs = (row["components_genesymbols"] or "").split("_")
    if subs == [""] or not set(subs) <= measured_syms:
        continue
    vids = [f"prot:{s}" for s in subs]
    members.update(vids)
    complex_specs.append({"edge_id": f'cpx:{row["name"]}', "edge_kind": "complex", "weight": 1.0,
                          "members": [(v, (uc.MECH_SIGNALING, uc.BASELINE)) for v in vids],
                          "complex_name": row["name"]})
G.add_vertices([{"vertex_id": v, "kind": uc.KIND_PROTEIN, "gene_symbol": v[len("prot:"):]}
                for v in sorted(members)], layer=(uc.MECH_SIGNALING, uc.BASELINE))
G.add_edges(complex_specs, layer=(uc.MECH_SIGNALING, uc.BASELINE))
G.history.snapshot("after_complex")
print(f"complexes kept: {len(complex_specs):,} | member proteins: {len(members):,}")

an.from_sbml(str(uc.HUMANGEM), graph=G, slice=uc.MECH_METABOLIC, layer=uc.METAB_COORD, preserve_stoichiometry=True)
V = G.views.vertices().to_pandas()
un = V[V["kind"].isna()]
G.attrs.set_vertex_attrs_bulk({v: {"kind": uc.KIND_METABOLITE}
                               for v in un.loc[~un.vertex_id.str.startswith("__"), "vertex_id"]})
G.attrs.set_vertex_attrs_bulk({v: {"kind": "boundary"}
                               for v in un.loc[un.vertex_id.str.startswith("__"), "vertex_id"]})
G.history.snapshot("after_metabolic")
print("baseline scaffold added | total edges:", G.global_count("edges"))

complexes kept: 7,991 | member proteins: 8,091


baseline scaffold added | total edges: 63891


In [5]:
G.write(str(uc.SNAPSHOT), overwrite=True)
print(f"saved {uc.SNAPSHOT}  |  V={G.global_count('vertices'):,}  E={G.global_count('edges'):,}")

G2 = an.AnnNet.read(str(uc.SNAPSHOT))
print(f"roundtrip: V={G2.global_count('vertices'):,} E={G2.global_count('edges'):,} "
      f"aspects={G2.layers.list_aspects()}")

saved data/uc2b.annnet  |  V=19,587  E=63,891


roundtrip: V=19,587 E=63,891 aspects=('mechanism', 'time')
